# 正式命名規則

## python3 ~/audit_chain_utils.py

> address：代表「這是哪一戶」（who）

> tag：代表「這是什麼類型的事件」（what）

household=A -> address=G9FIIBECECDDEDCAOJKAGBNKODNM9EHIAFKHCJDDHAINHNAAO9BIJHOHF9GFNBDL99999999999999999

---

household=B -> address=MEGIBKCLHIDCKLI9ILODHINLMJ9JJL9JDEHBNJK9KIIN9ONLDCNECGKJFOK9NCKC99999999999999999

---

household=C -> address=GCBLANKHCBHL99HCG9CECDG9FFDDONKBDABFJKNH9FACHDO9KHCDONIAHG9MFG9D99999999999999999

---

tag範例:
  unlock_door -> UNLOCKDOOR99999999999999999；
  view_camera -> VIEWCAMERA99999999999999999；
  unknown_action -> GENERICEVENT999999999999999

# 上鏈一筆稽核事件(送出交易)

## send_audit_event.py
iota@iota:~$ python3 ~/send_audit_event.py A unlock_door user_123 success
✅ 送出成功: household=A action=unlock_door actor=user_123 result=success
   address=G9FIIBECECDDEDCAOJKAGBNKODNM9E...
   tag=UNLOCKDOOR99999999999999999

---

iota@iota:~$ python3 ~/send_audit_event.py A view_camera user_123 success
✅ 送出成功: household=A action=view_camera actor=user_123 result=success
   address=G9FIIBECECDDEDCAOJKAGBNKODNM9E...
   tag=VIEWCAMERA99999999999999999

---

iota@iota:~$ python3 ~/send_audit_event.py B alert_triggered user_456 succes
s
✅ 送出成功: household=B action=alert_triggered actor=user_456 result=success
   address=MEGIBKCLHIDCKLI9ILODHINLMJ9JJL...
   tag=ALERTTRIG999999999999999999

# 跨屋隔離驗證

## python3 ~/audit_isolation_test.py

---

情境 1：家庭 A 住戶查詢自己家的資料

---

{'actual_household_queried': 'A', 'address_used': 'G9FIIBECECDDEDCAOJKAGBNKODNM9EHIAFKHCJDDHAINHNAAO9BIJHOHF9GFNBDL99999999999999999', 'multi_node_consensus': True, 'node_hash_counts': {'iri_1 (14265)': 2, 'node2 (14266)': 2, 'node3 (14267)': 2}, 'records': ['{"household":"A","actor":"user_123","action":"unlock_door","result":"success"}', '{"household":"A","actor":"user_123","action":"view_camera","result":"success"}']}

---

情境 2：家庭 A 住戶嘗試查詢家庭 B 的資料（應被攔截）

---

  [隔離攔截] user_A_resident 嘗試查詢 household=B，已被強制導回自己的 household=A
{'actual_household_queried': 'A', 'address_used': 'G9FIIBECECDDEDCAOJKAGBNKODNM9EHIAFKHCJDDHAINHNAAO9BIJHOHF9GFNBDL99999999999999999', 'multi_node_consensus': True, 'node_hash_counts': {'iri_1 (14265)': 2, 'node2 (14266)': 2, 'node3 (14267)': 2}, 'records': ['{"household":"A","actor":"user_123","action":"unlock_door","result":"success"}', '{"household":"A","actor":"user_123","action":"view_camera","result":"success"}']}

---

情境 3：Admin 查詢家庭 B 的資料（存取行為將自動上鏈）

---

  [Admin稽核] admin_1 查閱了 household=B 的資料，正在把這個存取行為本身上鏈...
✅ 送出成功: household=SYSTEM_AUDIT action=admin_access actor=admin_1 result=accessed_household:B
   address=MGMNBEIB9KLBOOBOAB9MHIADH9IOFM...
   tag=ADMINACCESS9999999999999999
{'actual_household_queried': 'B', 'address_used': 'MEGIBKCLHIDCKLI9ILODHINLMJ9JJL9JDEHBNJK9KIIN9ONLDCNECGKJFOK9NCKC99999999999999999', 'multi_node_consensus': True, 'node_hash_counts': {'iri_1 (14265)': 1, 'node2 (14266)': 1, 'node3 (14267)': 1}, 'records': ['{"household":"B","actor":"user_456","action":"alert_triggered","result":"success"}']}

等待5秒讓Admin存取紀錄透過P2P傳播...

---

查詢 SYSTEM_AUDIT：所有Admin存取行為的鏈上紀錄

---

  {"household":"SYSTEM_AUDIT","actor":"admin_1","action":"admin_access","result":"accessed_household:B"}